# 基于BERT的中文文本分类 - 外卖评论二分类（离线模式）

## 项目说明
本项目使用本地 bert-base-chinese 模型对 waimai_1k.csv 数据进行二分类训练，实现外卖评论的情感分类任务。

## 模型文件夹位置
模型保存在：`e:\BertTunning\bert-base-chinese\`

### 需要下载的文件：
- config.json（模型配置文件）
- pytorch_model.bin（预训练模型权重）
- vocab.txt（词汇表）
- tokenizer_config.json（分词器配置）

### 下载地址：
https://huggingface.co/bert-base-chinese/tree/main

---

## 项目结构
1. **环境配置**：导入必要的库和设置路径
2. **数据准备**：加载和预处理数据
3. **模型定义**：定义数据集类和BERT分类模型
4. **训练与评估**：训练模型并在测试集上评估

In [1]:
"""
==========================================
第一部分：导入必要的库
==========================================
本部分导入项目所需的所有Python库和模块
"""

# PyTorch相关库
import torch                    # PyTorch核心库，用于张量操作和自动求导
import torch.nn as nn           # 神经网络模块，用于定义模型层
from torch.utils.data import DataLoader, Dataset  # 数据加载器，用于批量处理数据

# 数据处理库
import pandas as pd             # 用于读取和处理CSV数据
import numpy as np            # 用于数值计算和数组操作

# 系统库
import os                       # 用于文件路径操作和检查文件是否存在

# Transformers库（Hugging Face）
from transformers import BertTokenizer, BertModel  # BERT分词器和模型

# 优化器和工具
from torch.optim import Adam    # Adam优化器，用于模型参数更新
from tqdm import tqdm           # 进度条显示，用于训练过程可视化

# 数据分割工具
from sklearn.model_selection import train_test_split  # 用于划分训练集、验证集和测试集

print("✅ 所有库导入完成")

c:\Users\19836\miniconda3\envs\py8\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 所有库导入完成


In [ ]:
"""
==========================================
第二部分：配置参数
==========================================
设置模型路径、超参数等配置信息
"""

# ========== 模型路径配置 ==========
# 设置本地 BERT 模型路径（离线模式）
# 注意：需要提前下载 bert-base-chinese 模型到该路径
BERT_MODEL_PATH = r'../bert-base-chinese'

# 检查模型文件夹是否存在
if not os.path.exists(BERT_MODEL_PATH):
    print(f"⚠️  警告：模型文件夹不存在: {BERT_MODEL_PATH}")
else:
    print(f"✅ 模型路径检查通过: {BERT_MODEL_PATH}")

# ========== 超参数配置 ==========
# 训练相关参数
EPOCHS = 3              # 训练轮数
LEARNING_RATE = 1e-5    # 学习率（建议范围：1e-5 到 5e-5）
BATCH_SIZE = 16         # 批次大小（根据GPU内存调整，CPU建议16-32）

# 模型相关参数
MAX_LENGTH = 512        # 文本最大长度（BERT最大支持512）
DROPOUT = 0.5           # Dropout比率，用于防止过拟合
NUM_CLASSES = 2         # 分类类别数（二分类：正面/负面）

# 数据分割比例
TRAIN_RATIO = 0.7       # 训练集比例
VAL_RATIO = 0.15        # 验证集比例
TEST_RATIO = 0.15       # 测试集比例

print(f"✅ 配置参数设置完成")
print(f"   - 训练轮数: {EPOCHS}")
print(f"   - 学习率: {LEARNING_RATE}")
print(f"   - 批次大小: {BATCH_SIZE}")

✅ 分词器加载完成


In [7]:
"""
==========================================
第三部分：初始化分词器
==========================================
加载BERT分词器，用于将中文文本转换为模型可理解的token序列
"""

# 从本地路径加载BERT分词器
# 分词器会将文本转换为：
# 1. input_ids: token的ID序列
# 2. attention_mask: 标记哪些位置是真实token（1）还是padding（0）
# 3. token_type_ids: 用于区分不同句子（单句分类任务中通常不需要）
tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_PATH)

print("✅ 分词器加载完成")
print(f"   - 词汇表大小: {len(tokenizer.vocab)}")
print(f"   - 特殊token示例: [CLS]={tokenizer.cls_token}, [SEP]={tokenizer.sep_token}, [PAD]={tokenizer.pad_token}")

数据总量: 1000
标签分布:
0    500
1    500
Name: label, dtype: int64

训练集大小: 700
验证集大小: 150
测试集大小: 150


In [8]:
"""
==========================================
第四部分：定义数据集类
==========================================
自定义Dataset类，用于将原始数据转换为模型可用的格式
"""

class TextDataset(Dataset):
    """
    文本分类数据集类
    
    功能：
    1. 将原始文本数据转换为BERT可接受的格式
    2. 对文本进行分词、padding和截断处理
    3. 返回处理后的文本和对应的标签
    """
    
    def __init__(self, df):
        """
        初始化数据集
        
        参数:
            df: pandas DataFrame，包含 'review' 和 'label' 列
        """
        # 将标签转换为numpy数组，避免pandas索引问题
        self.labels = df['label'].astype(int).values
        
        # 对每个文本进行分词处理
        # padding='max_length': 将文本填充到最大长度512
        # max_length=512: BERT支持的最大序列长度
        # truncation=True: 如果文本超过512，则截断
        # return_tensors="pt": 返回PyTorch张量格式
        self.texts = [
            tokenizer(
                text, 
                padding='max_length', 
                max_length=MAX_LENGTH, 
                truncation=True,
                return_tensors="pt"
            ) 
            for text in df['review']
        ]
        
    def __len__(self):
        """返回数据集大小"""
        return len(self.labels)

    def __getitem__(self, idx):
        """
        获取单个数据样本
        
        参数:
            idx: 数据索引
            
        返回:
            batch_texts: 处理后的文本（包含input_ids和attention_mask）
            batch_y: 对应的标签
        """
        batch_texts = self.get_batch_texts(idx)
        batch_y = self.get_batch_labels(idx)
        return batch_texts, batch_y
    
    def classes(self):
        """返回所有标签"""
        return self.labels
    
    def get_batch_labels(self, idx):
        """
        获取标签并转换为Long类型张量
        CrossEntropyLoss需要Long类型的标签
        """
        return torch.tensor(self.labels[idx], dtype=torch.long)

    def get_batch_texts(self, idx):
        """获取处理后的文本数据"""
        return self.texts[idx]

print("✅ 数据集类定义完成")

Some weights of the model checkpoint at e:\BertTunning\bert-base-chinese were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


✅ BERT 模型加载完成，开始训练...
使用设备: cpu


Epoch 1/3: 100%|██████████| 44/44 [14:13<00:00, 19.41s/it]


Epochs: 1 
              | Train Loss:  0.041 
              | Train Accuracy:  0.641 
              | Val Loss:  0.035 
              | Val Accuracy:  0.787


Epoch 2/3: 100%|██████████| 44/44 [16:27<00:00, 22.45s/it]


Epochs: 2 
              | Train Loss:  0.026 
              | Train Accuracy:  0.850 
              | Val Loss:  0.028 
              | Val Accuracy:  0.827


Epoch 3/3: 100%|██████████| 44/44 [14:46<00:00, 20.16s/it]


Epochs: 3 
              | Train Loss:  0.017 
              | Train Accuracy:  0.909 
              | Val Loss:  0.030 
              | Val Accuracy:  0.813


"""
==========================================
第五部分：定义BERT分类模型
==========================================
构建基于BERT的文本分类模型
"""

class BertClassifier(nn.Module):
    """
    BERT文本分类模型
    
    模型结构：
    1. BERT编码器：提取文本特征
    2. Dropout层：防止过拟合
    3. 全连接层：将768维特征映射到2个类别
    4. ReLU激活函数：增加非线性
    """
    
    def __init__(self, dropout=DROPOUT, model_path=BERT_MODEL_PATH):
        """
        初始化模型
        
        参数:
            dropout: Dropout比率，默认0.5
            model_path: BERT模型路径
        """
        super(BertClassifier, self).__init__()
        
        # 加载预训练的BERT模型（不包含分类头）
        # 使用本地模型路径，支持离线运行
        self.bert = BertModel.from_pretrained(model_path)
        
        # Dropout层：随机丢弃50%的神经元，防止过拟合
        self.dropout = nn.Dropout(dropout)
        
        # 全连接层：BERT输出维度是768，映射到2个类别（正面/负面）
        self.linear = nn.Linear(768, NUM_CLASSES)
        
        # ReLU激活函数：增加模型的非线性表达能力
        self.relu = nn.ReLU()

    def forward(self, input_id, mask):
        """
        前向传播
        
        参数:
            input_id: token ID序列，形状为 [batch_size, seq_length]
            mask: attention mask，形状为 [batch_size, seq_length]
            
        返回:
            final_layer: 分类结果，形状为 [batch_size, num_classes]
        """
        # 通过BERT模型获取文本表示
        # return_dict=False: 返回元组而不是字典
        # pooled_output: [CLS] token的表示，包含了整个句子的语义信息
        _, pooled_output = self.bert(
            input_ids=input_id, 
            attention_mask=mask,
            return_dict=False
        )
        
        # 应用Dropout
        dropout_output = self.dropout(pooled_output)
        
        # 通过全连接层得到分类logits
        linear_output = self.linear(dropout_output)
        
        # 应用ReLU激活函数
        final_layer = self.relu(linear_output)
        
        return final_layer

print("✅ BERT分类模型类定义完成")

In [10]:
"""
==========================================
第六部分：定义训练函数
==========================================
实现模型的训练和验证逻辑
"""

def train(model, train_data, val_data, learning_rate, epochs):
    """
    训练BERT分类模型
    
    参数:
        model: BertClassifier模型实例
        train_data: 训练数据DataFrame
        val_data: 验证数据DataFrame
        learning_rate: 学习率
        epochs: 训练轮数
    """
    # ========== 数据准备 ==========
    # 将DataFrame转换为Dataset对象
    train_dataset = TextDataset(train_data)
    val_dataset = TextDataset(val_data)
    
    # 创建DataLoader，用于批量加载数据
    # shuffle=True: 训练时打乱数据顺序，提高模型泛化能力
    train_dataloader = DataLoader(
        train_dataset, 
        batch_size=BATCH_SIZE,
        shuffle=True
    )
    val_dataloader = DataLoader(
        val_dataset, 
        batch_size=BATCH_SIZE
    )
    
    # ========== 设备配置 ==========
    # 检查是否有可用的GPU，如果有则使用GPU加速训练
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    print(f"使用设备: {device}")
    
    # ========== 损失函数和优化器 ==========
    # CrossEntropyLoss: 多分类交叉熵损失函数
    criterion = nn.CrossEntropyLoss()
    
    # Adam优化器：自适应学习率的优化算法
    optimizer = Adam(model.parameters(), lr=learning_rate)

    # 将模型和损失函数移到指定设备（CPU或GPU）
    if use_cuda:
        model = model.cuda()
        criterion = criterion.cuda()
    
    # ========== 训练循环 ==========
    print("\n开始训练...")
    for epoch_num in range(epochs):
        # ========== 训练阶段 ==========
        model.train()  # 设置为训练模式，启用Dropout等训练特性
        
        # 初始化累计指标
        total_acc_train = 0
        total_loss_train = 0
        
        # 遍历训练数据批次
        for train_input, train_label in tqdm(
            train_dataloader, 
            desc=f"Epoch {epoch_num + 1}/{epochs}"
        ):
            # 将数据移到指定设备
            train_label = train_label.to(device)
            mask = train_input['attention_mask'].to(device)
            input_id = train_input['input_ids'].squeeze(1).to(device)
            
            # 前向传播：通过模型得到预测结果
            output = model(input_id, mask)
            
            # 计算损失
            batch_loss = criterion(output, train_label)
            total_loss_train += batch_loss.item()
            
            # 计算准确率：预测类别与真实标签一致的数量
            acc = (output.argmax(dim=1) == train_label).sum().item()
            total_acc_train += acc
            
            # 反向传播和参数更新
            model.zero_grad()      # 清零梯度
            batch_loss.backward()  # 反向传播计算梯度
            optimizer.step()       # 更新模型参数
        
        # ========== 验证阶段 ==========
        model.eval()  # 设置为评估模式，禁用Dropout等训练特性
        
        # 初始化累计指标
        total_acc_val = 0
        total_loss_val = 0
        
        # 验证时不需要计算梯度，节省内存和计算资源
        with torch.no_grad():
            for val_input, val_label in val_dataloader:
                # 将数据移到指定设备
                val_label = val_label.to(device)
                mask = val_input['attention_mask'].to(device)
                input_id = val_input['input_ids'].squeeze(1).to(device)
                
                # 前向传播
                output = model(input_id, mask)
                
                # 计算损失和准确率
                batch_loss = criterion(output, val_label)
                total_loss_val += batch_loss.item()
                
                acc = (output.argmax(dim=1) == val_label).sum().item()
                total_acc_val += acc
        
        # ========== 输出训练结果 ==========
        print(
            f'''Epochs: {epoch_num + 1} 
              | Train Loss: {total_loss_train / len(train_dataset): .3f} 
              | Train Accuracy: {total_acc_train / len(train_dataset): .3f} 
              | Val Loss: {total_loss_val / len(val_dataset): .3f} 
              | Val Accuracy: {total_acc_val / len(val_dataset): .3f}'''
        )
    
    print("\n✅ 训练完成！")

print("✅ 训练函数定义完成")

测试中: 100%|██████████| 10/10 [00:54<00:00,  5.43s/it]

测试集 Loss:  0.026
测试集 Accuracy:  0.860


In [ ]:
"""
==========================================
第七部分：定义评估函数
==========================================
在测试集上评估模型性能
"""

def evaluate(model, test_data):
    """
    在测试集上评估模型性能
    
    参数:
        model: 训练好的BertClassifier模型
        test_data: 测试数据DataFrame
        
    返回:
        test_accuracy: 测试集准确率
    """
    # 创建测试数据集和数据加载器
    test_dataset = TextDataset(test_data)
    test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
    
    # 设备配置
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    
    # 设置为评估模式
    model.eval()
    
    # 初始化累计指标
    total_acc_test = 0
    total_loss_test = 0
    criterion = nn.CrossEntropyLoss()
    
    # 将模型和损失函数移到指定设备
    if use_cuda:
        model = model.cuda()
        criterion = criterion.cuda()
    
    # 在测试集上评估（不需要计算梯度）
    with torch.no_grad():
        for test_input, test_label in tqdm(test_dataloader, desc="测试中"):
            # 将数据移到指定设备
            test_label = test_label.to(device)
            mask = test_input['attention_mask'].to(device)
            input_id = test_input['input_ids'].squeeze(1).to(device)
            
            # 前向传播
            output = model(input_id, mask)
            
            # 计算损失和准确率
            batch_loss = criterion(output, test_label)
            total_loss_test += batch_loss.item()
            
            acc = (output.argmax(dim=1) == test_label).sum().item()
            total_acc_test += acc
    
    # 计算平均损失和准确率
    avg_loss = total_loss_test / len(test_dataset)
    avg_accuracy = total_acc_test / len(test_dataset)
    
    print(f"\n测试集评估结果:")
    print(f"  - Loss: {avg_loss: .3f}")
    print(f"  - Accuracy: {avg_accuracy: .3f}")
    
    return avg_accuracy

print("✅ 评估函数定义完成")

In [ ]:
"""
==========================================
第八部分：加载和预处理数据
==========================================
读取CSV文件，查看数据分布，并划分训练集、验证集和测试集
"""

# 读取CSV数据文件
# 数据文件应包含 'review'（评论文本）和 'label'（标签：0或1）两列
df = pd.read_csv('../waimai_1k.csv')

# 查看数据基本信息
print("=" * 50)
print("数据基本信息")
print("=" * 50)
print(f"数据总量: {len(df)}")
print(f"\n标签分布:")
print(df['label'].value_counts())
print(f"\n数据示例（前3条）:")
print(df.head(3))

# ========== 数据分割 ==========
# 使用分层抽样（stratify）确保训练集、验证集和测试集的标签分布一致
# 第一次分割：70%训练集，30%临时集
train_df, temp_df = train_test_split(
    df, 
    test_size=0.3, 
    stratify=df['label'],  # 分层抽样，保持标签比例
    random_state=42         # 随机种子，确保结果可复现
)

# 第二次分割：将30%的临时集分为15%验证集和15%测试集
val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    stratify=temp_df['label'],
    random_state=42
)

# 保存分割后的数据
_train = train_df
_val = val_df
_test = test_df

# 输出数据分割结果
print("\n" + "=" * 50)
print("数据分割结果")
print("=" * 50)
print(f"训练集大小: {len(_train)} ({len(_train)/len(df)*100:.1f}%)")
print(f"验证集大小: {len(_val)} ({len(_val)/len(df)*100:.1f}%)")
print(f"测试集大小: {len(_test)} ({len(_test)/len(df)*100:.1f}%)")

print("\n✅ 数据加载和预处理完成")

In [ ]:
"""
==========================================
第九部分：初始化模型
==========================================
创建BERT分类模型实例
"""

# 使用本地模型路径初始化分类器
# 模型会自动加载预训练的BERT权重
model = BertClassifier(model_path=BERT_MODEL_PATH)

# 输出模型信息
print("✅ BERT 模型初始化完成")
print(f"   - 模型参数数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"   - 可训练参数数量: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# 注意：如果看到关于未使用权重的警告，这是正常的
# 因为我们只使用BERT的编码器部分，而不需要预训练时的分类头

In [ ]:
"""
==========================================
第十部分：训练模型
==========================================
在训练集上训练模型，并在验证集上评估性能
"""

# 开始训练
# 训练过程会显示每个epoch的训练损失、训练准确率、验证损失和验证准确率
train(
    model=model,
    train_data=_train,
    val_data=_val,
    learning_rate=LEARNING_RATE,
    epochs=EPOCHS
)

In [ ]:
"""
==========================================
第十一部分：在测试集上评估模型
==========================================
使用训练好的模型在测试集上进行最终评估
注意：测试集在整个训练过程中都没有被使用，用于评估模型的真实泛化能力
"""

# 在测试集上评估模型
test_accuracy = evaluate(model, _test)

print(f"\n🎉 最终测试准确率: {test_accuracy:.3f}")

In [ ]:
## 训练完成！

### 模型性能总结
- 训练集：用于训练模型参数
- 验证集：用于调整超参数和监控训练过程
- 测试集：用于最终评估模型泛化能力

### 后续优化建议
1. **调整超参数**：尝试不同的学习率、批次大小、训练轮数
2. **数据增强**：增加训练数据量或使用数据增强技术
3. **模型微调**：尝试不同的dropout率或添加更多全连接层
4. **早停机制**：当验证集准确率不再提升时提前停止训练
5. **模型保存**：保存训练好的模型以便后续使用

### 保存模型示例代码
```python
# 保存模型
torch.save(model.state_dict(), 'bert_classifier.pth')

# 加载模型
model = BertClassifier()
model.load_state_dict(torch.load('bert_classifier.pth'))
model.eval()
```